# Лабораторная работа 15

Тема: **Transformer-энкодер для классификации тональности текстов**  
Формат: практическая работа с обязательными собственными экспериментами и комментариями.

> Этот ноутбук специально оформлен так, чтобы его нельзя было автоматически заполнить генеративной моделью без реального запуска экспериментов и анализа.  
> Каркас кода дан, но **основные баллы** ставятся за ваши настройки, графики и живые текстовые объяснения.


## 1. Ваше понимание архитектуры Transformer (если не знаетет, что такое GAN - можно гуглить и спрашивать LLM)

Перед запуском кода опишите текущее понимание (8–12 предложений):

1. В чем главное отличие Трансформеров от RNN (LSTM/GRU) при обработке последовательностей?
2. Зачем архитектуре, основанной исключительно на механизме внимания (Self-Attention), требуется Positional Encoding? Что будет, если его убрать?
3. Как вы интуитивно понимаете концепцию "нескольких голов" (Multi-Head Attention)? Зачем нужна разбивка на несколько независимых проекций?

Пишите своими словами, как если бы объясняли задачу одногруппнику.

In [1]:
intro_text = """1)Отличие Трансформеров от RNN при обработке последовательностей в том, что RNN обрабатывает последовательность шаг за шагом и вынуждена помнить всю историю в скрытом состоянии.
  Из-за этого её трудно ускорять, потому что каждый следующий шаг зависит от предыдущего.
  Transformer ализирует всю последовательность целиком, что позволяет учитывать контекст каждого элемента относительно всех остальных.

2)Positional Encoding в архитектурах, основанных исключительно на механизме внимания (Self-Attention) нужен для того, чтобы понимать последстовательсть слов, которая влияет на контекст.
Если его убрать, то предложения по типу 'слон наступил на мышь' будет ровно предложения 'мышь наступила на слона', то есть потеряется контекст предложения.

3)Multi-Head Attention позволяет модели одновременно улавливать разные типы взаимосвязей в одном слое, и обучение становится богаче и устойчивее.
Как говорится: одна голова - хорошо, а две - еше лучше!

"""
print(intro_text)

1)Отличие Трансформеров от RNN при обработке последовательностей в том, что RNN обрабатывает последовательность шаг за шагом и вынуждена помнить всю историю в скрытом состоянии.
  Из-за этого её трудно ускорять, потому что каждый следующий шаг зависит от предыдущего. 
  Transformer ализирует всю последовательность целиком, что позволяет учитывать контекст каждого элемента относительно всех остальных.

2)Positional Encoding в архитектурах, основанных исключительно на механизме внимания (Self-Attention) нужен для того, чтобы понимать последстовательсть слов, которая влияет на контекст.
Если его убрать, то предложения по типу 'слон наступил на мышь' будет ровно предложения 'мышь наступила на слона', то есть потеряется контекст предложения.

3)Multi-Head Attention позволяет модели одновременно улавливать разные типы взаимосвязей в одном слое, и обучение становится богаче и устойчивее.
Как говорится: одна голова - хорошо, а две - еше лучше!




## 2. Импорт, настройки и данные (IMDB)

Для работы потребуется библиотека `datasets`.
Если она не установлена, выполните `%pip install datasets`


In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset, concatenate_datasets
from collections import Counter
import math
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

MY_SEED = 42  # при своих экспериментах можете поменять, но зафиксируйте в отчёте
torch.manual_seed(MY_SEED)
np.random.seed(MY_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

# Загружаем датасет отзывов на фильмы
dataset = load_dataset("imdb")

vocab_size = 20000  # Максимальный размер словаря
max_seq_len = 256   # Максимальная длина отзыва / можно попробовать уменьшить для ускорения обучения
batch_size = 128

# 1. Строим словарь на обучающей выборке
counter = Counter()
for example in dataset['train']:
    # Базовая токенизация: приведение к нижнему регистру и разбиение по пробелам
    counter.update(example['text'].lower().split())

# 2. Оставляем самые частые слова и добавляем спецтокены
most_common = counter.most_common(vocab_size - 2)
vocab = {word: i + 2 for i, (word, _) in enumerate(most_common)}
vocab['<pad>'] = 0
vocab['<unk>'] = 1

PAD_IDX = 0
UNK_IDX = 1

def encode_text(text):
    """Преобразует строку в список индексов словаря"""
    return [vocab.get(word, UNK_IDX) for word in text.lower().split()]

def collate_batch(batch):
    """Функция для подготовки батча: обрезка и паддинг"""
    labels, texts = [], []
    for item in batch:
        labels.append(item['label'])
        encoded = torch.tensor(encode_text(item['text']), dtype=torch.int64)

        # Обрезаем слишком длинные отзывы
        if encoded.size(0) > max_seq_len:
            encoded = encoded[:max_seq_len]
        texts.append(encoded)

    labels = torch.tensor(labels, dtype=torch.int64)
    # Выравниваем длину текстов в батче, заполняя пустоты токеном <pad>
    texts = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)

    return texts, labels

# Создаем загрузчики данных. Здесь мы берём только часть примеров на трейн и тест (при вычислениях на ЦПУ одна эпоха займет примерно 3 минуты)
neg_dataset = dataset['train'].filter(lambda x: x['label'] == 0).select(range(2000))
pos_dataset = dataset['train'].filter(lambda x: x['label'] == 1).select(range(2000))

train_subset = concatenate_datasets([neg_dataset, pos_dataset]).shuffle(seed=MY_SEED)
test_subset = dataset['test'].shuffle(seed=MY_SEED).select(range(1000))

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

print("Размер словаря:", len(vocab))
print("Пример батча (shape):", next(iter(train_loader))[0].shape)

Устройство: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Размер словаря: 20000
Пример батча (shape): torch.Size([128, 256])


### Мини‑комментарий по предобработке данных

Кратко (3-5 предложений) опишите:

- какие проблемы возникают при приведении всех текстов к одной длине `max_seq_len` (padding и truncation) - что мы теряем и что приобретаем;
- зачем мы ввели токен `<unk>` (Unknown) и что произойдет, если модель встретит новое слово в тестовой выборке.


In [3]:
data_comment = """Приведение всех текстов к одной длине ускоряет процесс обучения.
Но при этом при слишком длинных отзывах будет обрезаться концовка, что приводит к потере важных слов, возможно, основной сути.
При слишком коротких отзывах появляются токены-заглушки, которые не несут в себе смысла.
Мы вводим токен <unk> вводится для редких слов или опечаток, то есть если слово не встречается в словаре, то ему присваивается этот токен для того, чтобы предсказание не ломалось.
То есть токен некоторая страховка от сбоев.
"""
print(data_comment)

Приведение всех текстов к одной длине ускоряет процесс обучения.
Но при этом при слишком длинных отзывах будет обрезаться концовка, что приводит к потере важных слов, возможно, основной сути.
При слишком коротких отзывах появляются токены-заглушки, которые не несут в себе смысла.
Мы вводим токен <unk> вводится для редких слов или опечаток, то есть если слово не встречается в словаре, то ему присваивается этот токен для того, чтобы предсказание не ломалось.
То есть токен некоторая страховка от сбоев.



## 3. Архитектура: Positional Encoding и Transformer


In [12]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x

class TextTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Классификатор (используем усредненный вектор по всей последовательности)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, src, src_key_padding_mask=None):
        # src shape: (batch_size, seq_len)
        embedded = self.embedding(src) * math.sqrt(d_model)
        embedded = self.pos_encoder(embedded)

        # output shape: (batch_size, seq_len, d_model)
        output = self.transformer_encoder(embedded, src_key_padding_mask=src_key_padding_mask)

        # Пулинг: берем среднее представление (только по реальным токенам, игнорируя PAD)
        # Для упрощения кода усредняем по всем выходам, но в продвинутых версиях нужно учитывать маску и при пулинге.
        pooled = output.mean(dim=1)

        return self.fc(pooled)

d_model = 128 # это можно менять
nhead = 4
num_layers = 2 # и это можно
num_classes = 2 # бинарная классификация (позитив/негатив)

model = TextTransformer(len(vocab), d_model, nhead, num_layers, num_classes).to(device)
print(model)

TextTransformer(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (pos_encoder): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


### Краткий анализ архитектуры

Ответьте в 4–6 предложениях:

- зачем эмбеддинги умножаются на `math.sqrt(d_model)` перед добавлением Positional Encoding;
- почему в качестве агрегации выходов Трансформера перед финальным `Linear` слоем мы используем усреднение `output.mean(dim=1)`, а не берем последний токен, как это делают в однонаправленных RNN.

In [13]:
arch_comment = """Умножение эмбеддингов необходим чтобы сбалансировать масштабы: сами эмбеддинги обычно инициализируются с небольшой дисперсией, а позиционные кодировки имеют амплитуду около 1.
Без этого множителя позиционный сигнал мог бы перебивать смысловое содержание слова.
Для агрегации мы берём среднее по всем токенам, а не последний, потому что Transformer не обрабатывает последовательность по порядку — каждая позиция видит сразу весь контекст, и «последний» токен ничем не выделен.
В отличие от однонаправленных RNN, где скрытое состояние последнего шага накапливает информацию обо всём предложении, в Transformer естественно учесть вклад всех позиций равноправно.
Усреднение — простой и эффективный способ получить единый вектор документа, не требующий введения специального классификационного токена.
"""
print(arch_comment)

Умножение эмбеддингов необходим чтобы сбалансировать масштабы: сами эмбеддинги обычно инициализируются с небольшой дисперсией, а позиционные кодировки имеют амплитуду около 1. 
Без этого множителя позиционный сигнал мог бы перебивать смысловое содержание слова.
Для агрегации мы берём среднее по всем токенам, а не последний, потому что Transformer не обрабатывает последовательность по порядку — каждая позиция видит сразу весь контекст, и «последний» токен ничем не выделен.
В отличие от однонаправленных RNN, где скрытое состояние последнего шага накапливает информацию обо всём предложении, в Transformer естественно учесть вклад всех позиций равноправно. 
Усреднение — простой и эффективный способ получить единый вектор документа, не требующий введения специального классификационного токена.



## 4. Оптимизатор и функция потерь


In [14]:
criterion = nn.CrossEntropyLoss()
lr = 2e-4
opt = torch.optim.Adam(model.parameters(), lr=lr)

## 5. Цикл обучения Трансформера

На каждой итерации вычисляем лосс, аккуратность (accuracy) и следим за метриками на тестовой выборке. Важный момент: мы создаем `padding_mask`, чтобы механизм Attention не обращал внимания на токены-пустышки.

In [15]:
def train_transformer(num_epochs):
    train_loss_hist, test_loss_hist = [], []
    train_acc_hist, test_acc_hist = [], []

    for epoch in range(1, num_epochs + 1):
        # --- Обучение ---
        model.train()
        epoch_loss, epoch_correct, total_samples = 0.0, 0, 0

        for x, y in tqdm(train_loader, desc=f"Обучение, Эпоха {epoch}/{num_epochs}"):
            x, y = x.to(device), y.to(device)

            # Маска для Attention: True там, где токен является паддингом
            mask = (x == PAD_IDX).to(device)

            opt.zero_grad()
            logits = model(x, src_key_padding_mask=mask)
            loss = criterion(logits, y)
            loss.backward()

            # Ограничение (clipping) градиентов для стабилизации обучения
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            epoch_loss += loss.item()
            preds = logits.argmax(dim=-1)
            epoch_correct += (preds == y).sum().item()
            total_samples += y.size(0)

        train_loss_hist.append(epoch_loss / len(train_loader))
        train_acc_hist.append(epoch_correct / total_samples)

        # --- Валидация ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                mask = (x == PAD_IDX).to(device)
                logits = model(x, src_key_padding_mask=mask)
                loss = criterion(logits, y)

                val_loss += loss.item()
                val_correct += (logits.argmax(dim=-1) == y).sum().item()
                val_total += y.size(0)

        test_loss_hist.append(val_loss / len(test_loader))
        test_acc_hist.append(val_correct / val_total)

        print(f"Эпоха {epoch}/{num_epochs} | "
              f"Train Loss: {train_loss_hist[-1]:.4f}, Acc: {train_acc_hist[-1]:.4f} | "
              f"Test Loss: {test_loss_hist[-1]:.4f}, Acc: {test_acc_hist[-1]:.4f}")

    return train_loss_hist, test_loss_hist, train_acc_hist, test_acc_hist

# Для тестов можно поставить 3-5 эпох, для полного обучения - 10-15
num_epochs = 10
train_loss, test_loss, train_acc, test_acc = train_transformer(num_epochs)

Обучение, Эпоха 1/10: 100%|██████████| 32/32 [00:02<00:00, 11.29it/s]


Эпоха 1/10 | Train Loss: 0.6827, Acc: 0.5437 | Test Loss: 0.6801, Acc: 0.5650


Обучение, Эпоха 2/10: 100%|██████████| 32/32 [00:02<00:00, 11.25it/s]


Эпоха 2/10 | Train Loss: 0.6044, Acc: 0.6887 | Test Loss: 0.6589, Acc: 0.6280


Обучение, Эпоха 3/10: 100%|██████████| 32/32 [00:02<00:00, 11.21it/s]


...

Эпоха 8/10 | Train Loss: 0.2111, Acc: 0.9223 | Test Loss: 0.7061, Acc: 0.6670


Обучение, Эпоха 9/10: 100%|██████████| 32/32 [00:02<00:00, 11.68it/s]


Эпоха 9/10 | Train Loss: 0.1585, Acc: 0.9467 | Test Loss: 0.7432, Acc: 0.6740


Обучение, Эпоха 10/10: 100%|██████████| 32/32 [00:02<00:00, 11.59it/s]


Эпоха 10/10 | Train Loss: 0.1062, Acc: 0.9712 | Test Loss: 0.7995, Acc: 0.6800


### Анализ кривых лоссов и метрик

Опишите:

- наблюдается ли на графиках переобучение (overfitting) Трансформера, и если да, то с какой эпохи (обратите внимание на разрыв между train_loss и test_loss);
- Трансформеры известны своей склонностью к переобучению при обучении «с нуля» на небольших наборах данных. Предложите 2 способа решения этой проблемы (помимо dropout, который уже используется).

In [16]:
loss_comment = """Графика нет.
Но исходя из вывода можно сказать, что где-то с 4 эпохи начинает наблюдаться переобучение.
train loss продолжает быстро снижаться и к 10-й доходит до 0.106, тогда как test loss после 4-й эпохи перестаёт уменьшаться и начинает расти (с 0.622 до 0.800).

Помимо dropout для решения переобучения можно попробовать использовать регуляризацию или замораживание и уменьшение числа слоёв/голов.
"""
print(loss_comment)

Графика нет. 
Но исходя из вывода можно сказать, что где-то с 4 эпохи начинает наблюдаться переобучение.
train loss продолжает быстро снижаться и к 10-й доходит до 0.106, тогда как test loss после 4-й эпохи перестаёт уменьшаться и начинает расти (с 0.622 до 0.800).

Помимо dropout для решения переобучения можно попробовать использовать регуляризацию или замораживание и уменьшение числа слоёв/голов.



## 7. Инференс: проверка модели на собственных текстах

Посмотрим, как модель классифицирует тексты, написанные вами.

In [11]:
def predict_sentiment(text):
    model.eval()
    encoded = torch.tensor([encode_text(text)], dtype=torch.int64).to(device)
    mask = (encoded == PAD_IDX).to(device)

    with torch.no_grad():
        logits = model(encoded, src_key_padding_mask=mask)
        prob = torch.softmax(logits, dim=-1)[0]

    pred_class = prob.argmax().item()
    label = "Positive" if pred_class == 1 else "Negative"
    print(f"Текст: {text}\nОценка: {label} (уверенность: {prob[pred_class].item():.4f})\n")

# Придумайте 2 явно позитивных отзыва и 2 явно негативных (на английском).
# Можно попробовать запутать модель сарказмом.
predict_sentiment("I absolutely loved this movie, it was fantastic!")
predict_sentiment("Great acting, wonderful plot, highly recommend.")

predict_sentiment("Terrible film, complete waste of time.")
predict_sentiment("Boring and badly written, I hated every minute.")

predict_sentiment("Oh brilliant, another masterpiece of predictable storytelling.")

Текст: I absolutely loved this movie, it was fantastic!
Оценка: Positive (уверенность: 0.5701)

Текст: Great acting, wonderful plot, highly recommend.
Оценка: Negative (уверенность: 0.5012)

Текст: Terrible film, complete waste of time.
Оценка: Negative (уверенность: 0.6465)

Текст: Boring and badly written, I hated every minute.
Оценка: Negative (уверенность: 0.5075)

Текст: Oh brilliant, another masterpiece of predictable storytelling.
Оценка: Negative (уверенность: 0.5411)



## 8. Идеи для вариаций в вашей работе

В **своём** варианте вы должны:

- попробовать вариации архитектуры: изменить размерность `d_model` (например, 64 и 256), изменить количество голов внимания `nhead` (например, 2 и 8), и сравнить, как это влияет на скорость сходимости и максимальную точность;
- поэкспериментировать с количеством слоев `num_layers` (1, 2, 4). Улучшается ли качество с добавлением глубины, или модель просто быстрее переобучается?
- вспомните (или посмотрите в свои старые записи), как на похожей задаче вела себя рекуррентная сеть (Лабораторная 10). Сравните Трансформер и LSTM по трем параметрам:
1) Скорость обучения (сколько секунд/минут уходило на эпоху при сопоставимом объеме данных).
2) Склонность к переобучению (кто быстрее начинает зубрить train и падать на val).
3) Итоговая точность (Accuracy).
- (Опционально) Изменить токенизатор, удалив стоп-слова и знаки препинания из исходного текста перед построением словаря, и оценить, помогло ли это поднять Accuracy.

In [ ]:
final_summary = """КОГДА ВЫ СДЕЛАЕТЕ СВОИ ВАРИАЦИИ ТРАНСФОРМЕРА (d_model,
nhead, num_layers), ИСПОЛЬЗУЙТЕ ЭТУ ЯЧЕЙКУ ДЛЯ ИТОГОВЫХ ВЫВОДОВ:
КАКАЯ КОНФИГУРАЦИЯ ОКАЗАЛАСЬ ОПТИМАЛЬНОЙ ДЛЯ ДАННОЙ ЗАДАЧИ, И
ОПРАВДАНО ЛИ ИСПОЛЬЗОВАНИЕ БОЛЬШОГО ЧИСЛА ГОЛОВ ВНИМАНИЯ ДЛЯ
АНАЛИЗА СМЫСЛА (СЕНТИМЕНТА).
В КАКОЙ РЕАЛЬНОЙ ЗАДАЧЕ ВЫ БЫ СЕЙЧАС ВЫБРАЛИ LSTM, А В КАКОЙ ТРАНСФОРМЕР?
у меня по полчаса и больше вычисления делаются, я не дождусь..."""
print(final_summary)

КОГДА ВЫ СДЕЛАЕТЕ СВОИ ВАРИАЦИИ ТРАНСФОРМЕРА (d_model, 
nhead, num_layers), ИСПОЛЬЗУЙТЕ ЭТУ ЯЧЕЙКУ ДЛЯ ИТОГОВЫХ ВЫВОДОВ: 
КАКАЯ КОНФИГУРАЦИЯ ОКАЗАЛАСЬ ОПТИМАЛЬНОЙ ДЛЯ ДАННОЙ ЗАДАЧИ, И 
ОПРАВДАНО ЛИ ИСПОЛЬЗОВАНИЕ БОЛЬШОГО ЧИСЛА ГОЛОВ ВНИМАНИЯ ДЛЯ 
АНАЛИЗА СМЫСЛА (СЕНТИМЕНТА).
В КАКОЙ РЕАЛЬНОЙ ЗАДАЧЕ ВЫ БЫ СЕЙЧАС ВЫБРАЛИ LSTM, А В КАКОЙ ТРАНСФОРМЕР?
